<a href="https://colab.research.google.com/github/cras-lab/ML-examples/blob/main/DeepLearning_Illustrated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 임포트 한다.

In [1]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

Seed를 동일하게 설정하여 재현가능하도록 한다.

In [ ]:
torch.manual_seed(0)

임의의 데이터를 생성한다. 곡선의 결정경계가 생기도록 생성한다.

In [3]:
n = 600

margin = 0.18
position_noise = 0.025
batch_size = 5000


def true_boundary(x1):
    return (
        0.45 * torch.sin(1.4 * x1)
        + 0.28 * torch.sin(3.1 * x1)
        + 0.16 * torch.sin(5.7 * x1)
        + 0.28 * torch.exp(-((x1 - 1.1) ** 2) / 0.18)
        - 0.25 * torch.exp(-((x1 + 1.5) ** 2) / 0.22)
    )


class0_points = []
class1_points = []


while (
    sum(p.shape[0] for p in class0_points) < n
    or sum(p.shape[0] for p in class1_points) < n
):
    x1 = torch.empty(batch_size).uniform_(-3.0, 3.0)
    x2 = torch.empty(batch_size).uniform_(-2.4, 2.4)

    points = torch.stack([x1, x2], dim=1)
    boundary = true_boundary(x1)

    mask0 = x2 < boundary - margin
    mask1 = x2 > boundary + margin

    if sum(p.shape[0] for p in class0_points) < n:
        class0_points.append(points[mask0])

    if sum(p.shape[0] for p in class1_points) < n:
        class1_points.append(points[mask1])


X0 = torch.cat(class0_points, dim=0)[:n]
X1 = torch.cat(class1_points, dim=0)[:n]

X0 = X0 + position_noise * torch.randn_like(X0)
X1 = X1 + position_noise * torch.randn_like(X1)

y0 = torch.zeros(n, 1)
y1 = torch.ones(n, 1)

X = torch.cat([X0, X1], dim=0)
y = torch.cat([y0, y1], dim=0)

perm = torch.randperm(len(X))
X = X[perm]
y = y[perm]

데이터를 그려본다.

In [ ]:
y_flat = y.squeeze()

plt.figure(figsize=(6, 5))

plt.scatter(
       X[y_flat == 0, 0],
       X[y_flat == 0, 1],
       c="blue",
       s=18,
       edgecolors="black",
       linewidths=0.4,
       label="class 0"
)

plt.scatter(
        X[y_flat == 1, 0],
        X[y_flat == 1, 1],
        c="red",
        s=18,
        edgecolors="black",
        linewidths=0.4,
        label="class 1"
)

plt.title("Data distribution")
plt.xlim(-3, 3)
plt.ylim(-3, 3)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


신경망을 생성하는 함수를 만들자.

In [5]:
def make_model(hidden_nodes, hidden_layers):
    layers = []

    input_dim = 2       # 시작은 입력이 2개 (2차원)

    for _ in range(hidden_layers):    # 은닉계층만큼 계층추가
        layers.append(nn.Linear(input_dim, hidden_nodes))
        layers.append(nn.ReLU())
        input_dim = hidden_nodes    # 그 다음 계층은 hidden_nodes

    layers.append(nn.Linear(input_dim, 1))    # 최종 입력은 1개

    return nn.Sequential(*layers)     #* 는 layers의 요소를 모두 풀어서 하나씩 해서 보냄

모델을 학습시키는 함수를 만들자.

In [6]:
def train_model(model, epochs=1000):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    losses = []

    for epoch in range(epochs):
        logits = model(X)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()       # 이전 단계에서 계산된 gradient를 지움
        loss.backward()             # parameter의 .grad에 gradient가 저장
        optimizer.step()            # gradient를 이용해서 weight 업데이트

        losses.append(loss.item())

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)
        pred = (probs > 0.5).float()
        acc = (pred == y).float().mean().item()

    return losses, acc

신경망 모양을 그려주는 함수를 만들자.

In [7]:
def draw_weighted_network(model, hidden_nodes, hidden_layers, acc):
    linear_layers = []

    for layer in model:
        if isinstance(layer, nn.Linear):
            linear_layers.append(layer)

    # 각 층의 노드 개수
    layer_sizes = [linear_layers[0].in_features]

    for layer in linear_layers:
        layer_sizes.append(layer.out_features)

    num_layers = len(layer_sizes)

    layer_names = ["Input"]

    for i in range(num_layers - 2):
        layer_names.append(f"Hidden {i + 1}")

    layer_names.append("Output")

    # 모든 weight 절댓값 모으기
    all_abs_weights = []

    for layer in linear_layers:
        w = layer.weight.detach().abs().reshape(-1)
        all_abs_weights.append(w)

    all_abs_weights = torch.cat(all_abs_weights)

    # max 하나가 너무 크면 나머지 선이 전부 너무 얇아지므로
    # 95% 분위수를 기준으로 두께 스케일링
    scale_weight = torch.quantile(all_abs_weights, 0.95).item()

    if scale_weight == 0:
        scale_weight = 1.0

    # 그림 크기
    plt.figure(figsize=(10, 6.5))

    x_positions = torch.linspace(0.08, 0.92, num_layers).tolist()
    node_positions = []

    # 노드 위치 계산
    for layer_idx, layer_size in enumerate(layer_sizes):
        x_pos = x_positions[layer_idx]

        if layer_size == 1:
            y_positions = [0.5]
        else:
            y_positions = torch.linspace(0.92, 0.08, layer_size).tolist()

        current_positions = []

        for y_pos in y_positions:
            current_positions.append((x_pos, y_pos))

        node_positions.append(current_positions)

    # ========================================================
    # 연결선 그리기
    # ========================================================

    for layer_idx, layer in enumerate(linear_layers):
        weight_matrix = layer.weight.detach().cpu()

        left_nodes = node_positions[layer_idx]
        right_nodes = node_positions[layer_idx + 1]

        # weight_matrix shape:
        # [다음 층 노드 개수, 이전 층 노드 개수]
        for right_idx, (x2, y2) in enumerate(right_nodes):
            for left_idx, (x1, y1) in enumerate(left_nodes):
                weight = weight_matrix[right_idx, left_idx].item()
                abs_weight = abs(weight)

                # 두께만 weight 크기를 반영
                normalized = abs_weight / scale_weight
                normalized = min(normalized, 1.0)

                # 최소 두께를 충분히 줘서 작은 weight도 보이게 함
                linewidth = 0.8 + 4.2 * normalized

                if weight >= 0:
                    linestyle = "-"
                    color = "royalblue"
                else:
                    linestyle = "--"
                    color = "crimson"

                plt.plot(
                    [x1, x2],
                    [y1, y2],
                    linestyle=linestyle,
                    linewidth=linewidth,
                    color=color,
                    alpha=0.85,
                    zorder=1
                )

    # ========================================================
    # 노드 그리기
    # ========================================================

    for layer_idx, layer in enumerate(node_positions):
        for node_idx, (x_pos, y_pos) in enumerate(layer):

            if layer_idx == 0:
                node_color = "lightgray"
                label = f"x{node_idx + 1}"
            elif layer_idx == num_layers - 1:
                node_color = "gold"
                label = "y"
            else:
                node_color = "white"
                label = ""

            plt.scatter(
                x_pos,
                y_pos,
                s=300,
                c=node_color,
                edgecolors="black",
                linewidths=1.6,
                zorder=3
            )

            if label:
                plt.text(
                    x_pos,
                    y_pos,
                    label,
                    ha="center",
                    va="center",
                    fontsize=10,
                    fontweight="bold",
                    zorder=4
                )

        plt.text(
            x_positions[layer_idx],
            1.035,
            f"{layer_names[layer_idx]}\n({layer_sizes[layer_idx]} nodes)",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold"
        )

    # ========================================================
    # 범례
    # ========================================================

    legend_items = [
        Line2D(
            [0], [0],
            color="royalblue",
            linewidth=3,
            linestyle="-",
            label="positive weight"
        ),
        Line2D(
            [0], [0],
            color="crimson",
            linewidth=3,
            linestyle="--",
            label="negative weight"
        ),
        Line2D(
            [0], [0],
            color="black",
            linewidth=1,
            linestyle="-",
            label="small |weight|"
        ),
        Line2D(
            [0], [0],
            color="black",
            linewidth=5,
            linestyle="-",
            label="large |weight|"
        )
    ]

    plt.legend(
        handles=legend_items,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=4,
        frameon=True
    )

    plt.title(
        f"Trained neural network weights | "
        f"nodes={hidden_nodes}, layers={hidden_layers}, acc={acc:.4f}",
        fontsize=13,
        fontweight="bold"
    )

    plt.xlim(0, 1)
    plt.ylim(0, 1.12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

결정 경계를 그리는 함수를 만들자.

In [8]:
def plot_decision_boundary(model, hidden_nodes, hidden_layers, acc):
    x_min, x_max = -3, 3
    y_min, y_max = -3, 3

    grid_x = torch.linspace(x_min, x_max, 250)
    grid_y = torch.linspace(y_min, y_max, 250)

    xx, yy = torch.meshgrid(grid_x, grid_y, indexing="ij")

    grid = torch.stack([
        xx.reshape(-1),
        yy.reshape(-1)
    ], dim=1)

    with torch.no_grad():
        logits = model(grid)
        probs = torch.sigmoid(logits)
        zz = probs.reshape(xx.shape)

    plt.figure(figsize=(6, 5))

    # 확률 배경
    plt.contourf(
        xx.numpy(),
        yy.numpy(),
        zz.numpy(),
        levels=50,
        cmap="coolwarm",
        alpha=0.45
    )

    # 실제 결정 경계: p = 0.5
    plt.contour(
        xx.numpy(),
        yy.numpy(),
        zz.numpy(),
        levels=[0.5],
        colors="black",
        linewidths=2
    )

    y_flat = y.squeeze()

    plt.scatter(
        X[y_flat == 0, 0].numpy(),
        X[y_flat == 0, 1].numpy(),
        c="blue",
        s=18,
        edgecolors="black",
        linewidths=0.4,
        label="class 0"
    )

    plt.scatter(
        X[y_flat == 1, 0].numpy(),
        X[y_flat == 1, 1].numpy(),
        c="red",
        s=18,
        edgecolors="black",
        linewidths=0.4,
        label="class 1"
    )

    plt.title(
        f"Decision boundary | "
        f"nodes={hidden_nodes}, layers={hidden_layers}, acc={acc:.4f}"
    )

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

전체 결과를 담아 둘 변수를 선언한다.

In [9]:
results = []

먼저 노드 2개와 1계층으로 시작해 보자.

In [10]:
hidden_nodes = 2
hidden_layers = 1

model = make_model(hidden_nodes, hidden_layers)
losses, acc = train_model(model, epochs=1000)
final_loss = losses[-1]

정확도와 손실을 출력해 보자.

In [11]:
print(f"accuracy={acc:.4f}, final_loss={final_loss:.4f}")

accuracy=0.9608, final_loss=0.0851


신경망 구조를 그려보자.

In [ ]:
draw_weighted_network(model, hidden_nodes, hidden_layers, acc)

In [ ]:
plot_decision_boundary(model, hidden_nodes, hidden_layers, acc)

나중에 전체 성능을 비교하기 위해 값을 저장해 둔다.

In [14]:
results.append({
        "hidden_nodes": hidden_nodes,
        "hidden_layers": hidden_layers,
        "accuracy": acc,
        "final_loss": final_loss
    })

계층을 그대로 두고 뉴런을 2==>4로 늘려보자.

In [15]:
hidden_nodes = 4
hidden_layers = 1

모델을 생성하고 훈련한다.

In [16]:
model = make_model(hidden_nodes, hidden_layers)
losses, acc = train_model(model, epochs=1000)
final_loss = losses[-1]

정확도와 손실값을 출력해 본다.

In [17]:
print(f"accuracy={acc:.4f}, final_loss={final_loss:.4f}")

accuracy=0.9608, final_loss=0.0874


신경망 모양을 출력해 보자.

In [ ]:
draw_weighted_network(model, hidden_nodes, hidden_layers, acc)

결정경계를 그려보자.

In [ ]:
plot_decision_boundary(model, hidden_nodes, hidden_layers, acc)

나중에 전체 성능을 비교하기 위해 값을 저장해 둔다.

In [20]:
results.append({
        "hidden_nodes": hidden_nodes,
        "hidden_layers": hidden_layers,
        "accuracy": acc,
        "final_loss": final_loss
    })

계층을 그대로 두고 뉴런을 4==>8로 늘려보자.

In [21]:
hidden_nodes = 8
hidden_layers = 1

모델을 생성하고 훈련한다.

In [22]:
model = make_model(hidden_nodes, hidden_layers)
losses, acc = train_model(model, epochs=1000)
final_loss = losses[-1]

정확도와 손실값을 출력해 본다.

In [23]:
print(f"accuracy={acc:.4f}, final_loss={final_loss:.4f}")

accuracy=0.9975, final_loss=0.0125


신경망 모양을 출력해 보자.

In [ ]:
draw_weighted_network(model, hidden_nodes, hidden_layers, acc)

결정경계를 그려보자.

In [ ]:
plot_decision_boundary(model, hidden_nodes, hidden_layers, acc)

나중에 전체 성능을 비교하기 위해 값을 저장해 둔다.

In [26]:
results.append({
        "hidden_nodes": hidden_nodes,
        "hidden_layers": hidden_layers,
        "accuracy": acc,
        "final_loss": final_loss
    })

계층을 2개로 늘려보자. 뉴런도 16개로 늘려보자.

In [27]:
hidden_nodes = 16
hidden_layers = 2

모델을 생성하고 훈련한다.

In [28]:
model = make_model(hidden_nodes, hidden_layers)
losses, acc = train_model(model, epochs=1000)
final_loss = losses[-1]

정확도와 손실값을 출력해 본다.

In [29]:
print(f"accuracy={acc:.4f}, final_loss={final_loss:.4f}")

accuracy=1.0000, final_loss=0.0012


신경망 모양을 출력해 보자.

In [ ]:
draw_weighted_network(model, hidden_nodes, hidden_layers, acc)

결정경계를 그려보자.

In [ ]:
plot_decision_boundary(model, hidden_nodes, hidden_layers, acc)

나중에 전체 성능을 비교하기 위해 값을 저장해 둔다.

In [32]:
results.append({
        "hidden_nodes": hidden_nodes,
        "hidden_layers": hidden_layers,
        "accuracy": acc,
        "final_loss": final_loss
    })

계층을 2개로 두고 뉴런을 32개로 늘려보자.

In [33]:
hidden_nodes = 32
hidden_layers = 2

모델을 생성하고 훈련한다.

In [34]:
model = make_model(hidden_nodes, hidden_layers)
losses, acc = train_model(model, epochs=1000)
final_loss = losses[-1]

정확도와 손실값을 출력해 본다.

In [ ]:
print(f"accuracy={acc:.4f}, final_loss={final_loss:.4f}")

신경망 모양을 출력해 보자.

In [ ]:
draw_weighted_network(model, hidden_nodes, hidden_layers, acc)

결정경계를 그려보자.

In [ ]:
plot_decision_boundary(model, hidden_nodes, hidden_layers, acc)

나중에 전체 성능을 비교하기 위해 값을 저장해 둔다.

In [38]:
results.append({
        "hidden_nodes": hidden_nodes,
        "hidden_layers": hidden_layers,
        "accuracy": acc,
        "final_loss": final_loss
    })

최종 결과를 종합해서 출력해 보자.

In [ ]:
for r in results:
    print(
        f"nodes={r['hidden_nodes']:>2}, "
        f"layers={r['hidden_layers']}, "
        f"accuracy={r['accuracy']:.4f}, "
        f"final_loss={r['final_loss']:.4f}"
    )